Extending the Tiktoken BPE Tokenizer with New Tokens

In [ ]:
# 导入 tiktoken 库（OpenAI 开源的高性能 BPE 分词器实现，GPT-2/GPT-3/GPT-4 等模型都用它）
import tiktoken

# 加载 GPT-2 官方使用的基础编码器（未做任何扩展的原始 BPE 分词器）
base_tokenizer = tiktoken.get_encoding("gpt2")
# 构造一段示例文本，其中 MyNewToken_1 是我们后面想让分词器认识的自定义 token
sample_text = "Hello, MyNewToken_1 is a new token. <|endoftext|>"

# 用原始分词器编码；allowed_special 显式放行 <|endoftext|>（否则遇到特殊 token 默认会报错）
# 此时 MyNewToken_1 还不是特殊 token，会被按普通文本拆成多个 BPE 子词
token_ids = base_tokenizer.encode(sample_text, allowed_special={"<|endoftext|>"})
print(token_ids)

1. Adding special tokens

In [ ]:
# 逐个打印 token id 及其解码后的文本片段，可以看到 MyNewToken_1 被拆成了多个子词 token
for token_id in token_ids:
    print(f"{token_id} -> {base_tokenizer.decode([token_id])}")

In [ ]:
# Define custom tokens and their token IDs
# 定义想要新增的自定义特殊 token 名称列表
custom_tokens = ["MyNewToken_1", "MyNewToken_2"]
# 为每个自定义 token 分配新的 id：从原始词表大小 base_tokenizer.n_vocab 开始依次往后编号，
# 这样可以保证新 id 不会与原有词表中任何一个 token id 冲突
custom_token_ids = {
    token: base_tokenizer.n_vocab + i for i, token in enumerate(custom_tokens)
}

In [ ]:
# Create a new Encoding object with extended tokens
# 基于原始分词器手动构造一个新的 tiktoken.Encoding 对象，实现"扩展词表"
extended_tokenizer = tiktoken.Encoding(
    name="gpt2_custom",
    # pat_str：用于预分词（pre-tokenization）的正则表达式，直接复用 GPT-2 原有的切分规则
    pat_str=base_tokenizer._pat_str,
    # mergeable_ranks：BPE 合并规则表（即普通词表本身），同样完全复用，不做任何修改
    mergeable_ranks=base_tokenizer._mergeable_ranks,
    # special_tokens：用字典解包合并——保留原有的特殊 token（如 <|endoftext|>），
    # 再加入上面新定义的 custom_token_ids，从而"追加"出新的特殊 token
    # 注意：pat_str / mergeable_ranks / _special_tokens 都是 tiktoken.Encoding 的私有属性（下划线开头），
    # 依赖具体 tiktoken 版本的内部实现，未来版本升级可能导致属性名变化或不可用——这是一个风险点，仅标注，不做修改
    special_tokens={**base_tokenizer._special_tokens, **custom_token_ids},
)

In [ ]:
# 汇总所有需要在 encode 时显式放行的特殊 token 名称：自定义 token 并上原有的 <|endoftext|>
special_tokens_set = set(custom_tokens) | {"<|endoftext|>"}

# 用扩展后的分词器编码文本；allowed_special 传入上面这个集合，
# 使 MyNewToken_1 / MyNewToken_2 / <|endoftext|> 都被当作完整的单个特殊 token 处理，而不再被继续拆分
token_ids = extended_tokenizer.encode(
    "Sample text with MyNewToken_1 and MyNewToken_2. <|endoftext|>",
    allowed_special=special_tokens_set
)
print(token_ids)


In [ ]:
# 打印新分词器的编码结果：可以看到 MyNewToken_1 / MyNewToken_2 各自对应一个完整的单一 token id，
# 不再像之前那样被拆成多个子词
for token_id in token_ids:
    print(f"{token_id} -> {extended_tokenizer.decode([token_id])}")

2. Updating a pretrained LLM
In this section, we will take a look at how we have to update an existing pretrained LLM after updating the tokenizer
For this, we are using the original pretrained GPT-2 model that is used in the main book


2.1 Loading a pretrained GPT model

In [ ]:
# 从 llms_from_scratch 包导入下载并加载官方 GPT-2 预训练权重的工具函数
# （import 路径已是正确的 llms_from_scratch，非常见的 Build_an_LLM_from_Scratch 误写，无需修改）
from llms_from_scratch.ch05 import download_and_load_gpt2
# For llms_from_scratch installation instructions, see:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg

# 下载（若本地无缓存）并加载 124M 参数规模的 GPT-2 预训练模型，
# 返回超参数配置 settings 和权重字典 params
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
# 从 llms_from_scratch 包导入本书实现的 GPTModel 类
# （import 路径已是正确的 llms_from_scratch，无需修改）
from llms_from_scratch.ch04 import GPTModel
# For llms_from_scratch installation instructions, see:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg

# 124M 模型的基础超参数配置（这里的 context_length 先用一个较短的示例值，后面会被覆盖为完整值）
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

# Define model configurations in a dictionary for compactness
# 把不同规模 GPT-2（small/medium/large/xl）各自的 emb_dim、n_layers、n_heads 汇总到一个字典里，
# 方便按名称一键切换模型规模
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
# 复制基础配置（避免直接修改 GPT_CONFIG_124M 这个原始字典）
NEW_CONFIG = GPT_CONFIG_124M.copy()
# 用所选模型规模对应的 emb_dim / n_layers / n_heads 覆盖基础配置里的同名字段
NEW_CONFIG.update(model_configs[model_name])
# 再把 context_length 覆盖为完整的 1024（与预训练权重的位置编码长度对齐），
# 并开启 qkv_bias（原始 GPT-2 权重里 Q/K/V 线性层带有偏置项，需要保持一致才能正确加载权重）
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

# 用最终的配置实例化 GPTModel
gpt = GPTModel(NEW_CONFIG)
# 切换为 eval 模式（关闭 dropout 等训练态行为）；末尾分号用于抑制 Jupyter 对返回值的自动打印
gpt.eval();

2.2 Using the pretrained GPT model

In [ ]:
# 重新定义一段包含自定义 token 的示例文本，用于对比两种分词器编码结果的差异
sample_text = "Sample text with MyNewToken_1 and MyNewToken_2. <|endoftext|>"

# 用原始（未扩展）分词器编码：MyNewToken_1/2 不被识别为特殊 token，会被拆成多个普通子词
original_token_ids = base_tokenizer.encode(
    sample_text, allowed_special={"<|endoftext|>"}
)
# 用扩展后的分词器编码同一段文本：MyNewToken_1/2 被当作完整的单一特殊 token id 处理
# 两组 token id 的长度和内容都会不同，后面用于对比模型在两种输入下的表现
new_token_ids = extended_tokenizer.encode(
    "Sample text with MyNewToken_1 and MyNewToken_2. <|endoftext|>",
    allowed_special=special_tokens_set
)

In [ ]:
# 导入 PyTorch
import torch

# 用 torch.no_grad() 关闭梯度计算（纯推理场景，节省显存和计算开销）
with torch.no_grad():
    # 把 token id 列表包装成 batch 维度为 1 的张量后送入模型做前向传播，得到每个位置在整个词表上的 logits
    # 此时模型仍是原始未扩容的 GPT-2，而 original_token_ids 里也不包含超出原词表范围的 id，因此可以正常运行
    out = gpt(torch.tensor([original_token_ids]))

print(out)

2.3 Updating the embedding layer

In [ ]:
# 查看当前模型的词嵌入层（nn.Embedding），确认其形状仍是原始词表大小（50257）x emb_dim，
# 还没有为新增的两个自定义 token 留出对应的嵌入向量位置
gpt.tok_emb

In [ ]:
# 读取原嵌入层的形状：num_tokens 是原词表大小，emb_size 是每个 token 的嵌入维度
num_tokens, emb_size = gpt.tok_emb.weight.shape
# 新词表大小 = 原词表大小 + 新增的 2 个自定义 token
new_num_tokens = num_tokens + 2

# Create a new embedding layer
# 新建一个行数更多的嵌入层，行数扩展为 new_num_tokens，其余（列数/维度）保持不变
new_embedding = torch.nn.Embedding(new_num_tokens, emb_size)

# Copy weights from the old embedding layer
# 把原嵌入层的权重原样拷贝到新嵌入层的前 num_tokens 行；新增的最后 2 行沿用 nn.Embedding 的默认随机初始化，
# 对应两个新 token，需要后续微调训练才能学到有意义的表示
new_embedding.weight.data[:num_tokens] = gpt.tok_emb.weight.data

# Replace the old embedding layer with the new one in the model
# 用扩容后的嵌入层替换模型中原来的 tok_emb 属性
gpt.tok_emb = new_embedding

print(gpt.tok_emb)

2.4 Updating the output layer

In [ ]:
# 查看当前输出层（nn.Linear，负责把隐藏状态映射回词表大小的 logits），
# 确认其输出维度同样还是原始词表大小，尚未扩容
gpt.out_head

In [ ]:
# nn.Linear 的权重形状是 [out_features, in_features]，
# 这里取出原始输出维度（等于词表大小）和输入维度（等于 emb_size）
original_out_features, original_in_features = gpt.out_head.weight.shape

# Define the new number of output features (e.g., adding 2 new tokens)
# 新的输出维度同样加 2，对应新增的两个自定义 token
new_out_features = original_out_features + 2

# Create a new linear layer with the extended output size
# 创建一个输入维度不变、输出维度扩大的新线性层
new_linear = torch.nn.Linear(original_in_features, new_out_features)

# Copy the weights and biases from the original linear layer
# 在 no_grad 上下文中手动拷贝参数，避免被记录进 autograd 计算图
with torch.no_grad():
    # 把原输出层的权重拷贝到新层对应的前 original_out_features 行
    new_linear.weight[:original_out_features] = gpt.out_head.weight
    # 如果原层带 bias，也同样拷贝过去；新增的 2 行权重/偏置沿用 nn.Linear 的默认初始化
    if gpt.out_head.bias is not None:
        new_linear.bias[:original_out_features] = gpt.out_head.bias

# Replace the original linear layer with the new one
# 用扩容后的线性层替换模型的输出头 out_head
gpt.out_head = new_linear

print(gpt.out_head)

In [ ]:
# 用扩容后的模型，对"原始（未扩展）分词器"编码得到的 token id 序列做前向传播，
# 验证模型在旧的 token 分布上依然可以正常工作（此时输出 logits 的最后一维已经是扩容后的新词表大小）
with torch.no_grad():
    output = gpt(torch.tensor([original_token_ids]))
print(output)

In [ ]:
# 用扩容后的模型，对"扩展分词器"编码得到的 token id（包含新增的 MyNewToken_1/2 id）做前向传播；
# 因为嵌入层和输出层都已经扩容，即便 token id 超出了原始词表范围也能正常索引，不会报错
with torch.no_grad():
    output = gpt(torch.tensor([new_token_ids]))
print(output)

In [ ]:
# 让输出层与输入嵌入层共享同一份权重张量（weight tying，GPT-2 原始实现中采用的做法），
# 可以减少参数量并保持输入/输出嵌入语义的一致性；
# 注意：这里的赋值会把 out_head.weight 直接替换成 tok_emb.weight 这同一个 Parameter 对象，
# 之后两者会共用同一份存储，训练时会一起被更新
gpt.out_head.weight = gpt.tok_emb.weight

In [ ]:
# 权重共享（weight tying）之后再跑一次前向传播，验证共享权重后模型依然能正常工作
# （注意这里没有 print 语句，如需查看结果需要另行打印 output）
with torch.no_grad():
    output = gpt(torch.tensor([new_token_ids]))